In [ ]:
from general_utils import find_ephys_sessions
from nwb_utils import NWBUtils
from behavior_utils import get_fitted_model_names, get_fitted_latent
_, _, sessions = find_ephys_sessions()
from behavior_qc_visualization import collect_behavior_model_summary
summary=collect_behavior_model_summary(sessions=['ecephys_795393_2025-09-15_13-05-25_sorted_2025-10-29_20-10-36'])

In [ ]:
from general_utils import find_ephys_sessions
from nwb_utils import NWBUtils
from behavior_utils import get_fitted_model_names, get_fitted_latent
_, _, sessions = find_ephys_sessions()
from behavior_qc_visualization import collect_behavior_model_summary
summary=collect_behavior_model_summary(sessions=sessions)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd

from behavior_qc_visualization import collect_behavior_model_summary

def _run_one_session(session_name: str) -> pd.DataFrame:
    # If collect_behavior_model_summary can accept a single-session list, reuse it.
    # This keeps behavior identical to your existing function.
    df = collect_behavior_model_summary(sessions=[session_name])
    # Optional: add session column if it's not already present
    if "session_name" not in df.columns:
        df = df.assign(session_name=session_name)
    return df

def collect_behavior_model_summary_parallel(
    sessions,
    *,
    max_workers: int = 8,
) -> pd.DataFrame:
    dfs = []
    errors = []

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        fut_to_sess = {ex.submit(_run_one_session, s): s for s in sessions}
        for fut in as_completed(fut_to_sess):
            s = fut_to_sess[fut]
            try:
                dfs.append(fut.result())
            except Exception as e:
                errors.append((s, repr(e)))

    if errors:
        # Don't crash everything; return what you got + a helpful report
        print("Some sessions failed:")
        for s, err in errors[:20]:
            print(f"  {s}: {err}")
        if len(errors) > 20:
            print(f"  ... plus {len(errors)-20} more")

    if not dfs:
        return pd.DataFrame()

    out = pd.concat(dfs, ignore_index=True, sort=False)

    # Optional: stable ordering if you want
    if "session_name" in out.columns:
        out = out.sort_values(["session_name"]).reset_index(drop=True)

    return out

# Usage
summary = collect_behavior_model_summary_parallel(sessions, max_workers=8)


In [11]:
summary.columns

Index(['session', 'QLearning_L1F1_CK1_softmax_learn_rate_rew',
       'QLearning_L1F1_CK1_softmax_learn_rate_unrew',
       'QLearning_L1F1_CK1_softmax_forget_rate_unchosen',
       'QLearning_L1F1_CK1_softmax_choice_kernel_relative_weight',
       'QLearning_L1F1_CK1_softmax_choice_kernel_step_size',
       'QLearning_L1F1_CK1_softmax_biasL',
       'QLearning_L1F1_CK1_softmax_softmax_inverse_temperature',
       'QLearning_L1F1_CK1_softmax_learn_rate',
       'QLearning_L1F1_CK1_softmax_log_likelihood',
       'QLearning_L1F1_CK1_softmax_AIC', 'QLearning_L1F1_CK1_softmax_BIC',
       'QLearning_L1F1_CK1_softmax_LPT', 'QLearning_L1F1_CK1_softmax_LPT_AIC',
       'QLearning_L1F1_CK1_softmax_LPT_BIC',
       'QLearning_L1F1_CK1_softmax_prediction_accuracy',
       'QLearning_L1F1_CK1_softmax_reward_coefs',
       'QLearning_L2F1_softmax_learn_rate_rew',
       'QLearning_L2F1_softmax_learn_rate_unrew',
       'QLearning_L2F1_softmax_forget_rate_unchosen',
       'QLearning_L2F1_softmax_

In [12]:
summary['ForagingCompareThreshold_biasL']

0    -0.121772
1     0.944088
2     1.489988
3     1.112281
4     1.540877
5    -0.390170
6    -0.129554
7    -0.027514
8    -0.729377
9    -1.618812
10   -1.566551
11    0.283636
12    0.724254
13   -0.087863
14   -0.431513
15   -0.243750
16   -0.410170
17    0.321080
18   -0.170318
19   -0.398214
20   -0.189941
21   -0.405307
22         NaN
23   -0.205762
24   -0.055082
25    0.291562
26    0.248919
27   -0.349725
28   -0.525920
29    0.321291
30   -0.338922
31   -0.093578
32   -0.113159
33   -0.760637
34   -0.340023
35    0.005720
36    0.073474
37    0.491517
38    0.195276
39    0.421801
40   -0.531745
Name: ForagingCompareThreshold_biasL, dtype: float64